# 02 Clean Donated YouTube Search Data

This notebook demonstrates the second cleaning step in the data-donation workflow: turning raw Google Takeout-style YouTube search-history files into a combined search-history table and a narrower table of genuine YouTube search queries.

The notebook uses mock donor data stored in `data/mock_takeout`. The same logic can be adapted to confidential GDPR data donations by changing the input path and adding project-specific consent, exclusion, and participant metadata rules.

## What This Notebook Produces

The notebook writes two CSV files to `outputs/tables`.

- `search_histories.csv`: all donated search-history rows combined and standardized.
- `search_query_histories.csv`: only rows classified as genuine YouTube search queries.

The second table is the handoff for later notebooks that estimate whether watched videos occurred shortly after a participant searched on YouTube.

In [ ]:
from pathlib import Path
import re
from urllib.parse import parse_qs, urlparse

import pandas as pd

## Locate The Project And Input Data

The notebook is designed to run from the Quarto project root or from the `scripts` folder. The helper below walks upward from the current working directory until it finds the Quarto project and the mock Takeout folder.

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        has_quarto_project = (candidate / "_quarto.yml").exists()
        has_mock_data = (candidate / "data" / "mock_takeout").exists()
        if has_quarto_project and has_mock_data:
            return candidate
    raise FileNotFoundError(
        "Could not find the Quarto project root. Run this notebook from "
        "inside youtube_donation_dsa_method."
    )


PROJECT_ROOT = find_project_root()
MOCK_TAKEOUT_DIR = PROJECT_ROOT / "data" / "mock_takeout"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

SEARCH_HISTORIES_PATH = OUTPUT_DIR / "search_histories.csv"
SEARCH_QUERY_HISTORIES_PATH = OUTPUT_DIR / "search_query_histories.csv"

print(f"Project folder: {PROJECT_ROOT.name}")
print(f"Mock data folder: {MOCK_TAKEOUT_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Output folder: {OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix()}")

## Find Donated Search-History Files

Google Takeout stores YouTube search history at `YouTube and YouTube Music/history/search-history.json`. In this mock dataset, each donor has one folder, so the donor folder name becomes the participant identifier.

In [ ]:
search_files = sorted(
    MOCK_TAKEOUT_DIR.glob("*/YouTube and YouTube Music/history/search-history.json")
)

search_file_index = pd.DataFrame(
    {
        "Participant ID": [path.relative_to(MOCK_TAKEOUT_DIR).parts[0] for path in search_files],
        "search_history_file": [path.relative_to(PROJECT_ROOT).as_posix() for path in search_files],
    }
)

search_file_index

## Load And Combine Search Histories

Each JSON file is loaded into a dataframe, annotated with `Participant ID`, and then combined into one raw search-history table. At this point, the table can include genuine search queries as well as other rows that appear in the search-history export.

In [ ]:
def load_search_history(path):
    participant_id = path.relative_to(MOCK_TAKEOUT_DIR).parts[0]
    data = pd.read_json(path)
    data["Participant ID"] = participant_id
    return data


search_histories = pd.concat(
    [load_search_history(path) for path in search_files],
    ignore_index=True,
)

search_histories.shape

## Standardize Columns

The original Takeout fields are useful but not always named for analysis. We rename `title` to `search_title` and `titleUrl` to `url`. We keep `details` and `description` in the combined table so non-query rows remain inspectable without assigning them stronger labels than the data support.

In [ ]:
search_histories = search_histories.rename(
    columns={
        "title": "search_title",
        "titleUrl": "url",
    }
)

core_columns = [
    "Participant ID",
    "time",
    "header",
    "search_title",
    "url",
    "details",
    "description",
]

search_histories = search_histories.reindex(columns=core_columns)
search_histories.head()

## Parse Timestamps

Takeout timestamps are ISO-formatted UTC strings. Some rows include milliseconds and some do not, so we use pandas' ISO8601 parser explicitly.

In [ ]:
search_histories["time"] = pd.to_datetime(
    search_histories["time"],
    format="ISO8601",
    utc=True,
)

search_histories["time"].agg(["min", "max"])

## Decode Search Queries From URLs

YouTube search-history rows point to search-results URLs such as `https://www.youtube.com/results?search_query=demo+search+query`. Decoding this URL gives a language-neutral query field that later notebooks can use for temporal matching.

In [ ]:
def extract_query_from_url(url):
    if not isinstance(url, str):
        return pd.NA
    parsed = urlparse(url)
    is_youtube_results_url = (
        parsed.scheme == "https"
        and parsed.netloc == "www.youtube.com"
        and parsed.path == "/results"
    )
    if not is_youtube_results_url:
        return pd.NA
    query_values = parse_qs(parsed.query, keep_blank_values=False).get("search_query", [])
    if not query_values or not query_values[0].strip():
        return pd.NA
    return query_values[0].strip()


search_histories["query_from_url"] = search_histories["url"].apply(extract_query_from_url)
search_histories["has_standard_search_url"] = search_histories["query_from_url"].notna()

search_histories[["search_title", "url", "query_from_url"]].head()

## Identify Multilingual Search Actions

Google localizes the action phrase in Takeout titles according to the donor account language. The original workflow identified genuine search rows by extracting an action phrase and retaining known search actions such as `Searched`, `Søgte`, `Hai cercato`, `Szukano`, and `Выполнен поиск`.

We keep that logic explicit here and add transparent columns so the classification can be audited.

In [ ]:
SEARCH_ACTIONS = pd.DataFrame(
    [
        {"search_action": "Søgte", "language_hint": "Danish/Norwegian"},
        {"search_action": "Searched", "language_hint": "English"},
        {"search_action": "Hai cercato", "language_hint": "Italian"},
        {"search_action": "Szukano", "language_hint": "Polish"},
        {"search_action": "Выполнен поиск", "language_hint": "Russian"},
    ]
)

known_search_actions = set(SEARCH_ACTIONS["search_action"])

SEARCH_ACTIONS

In [ ]:
def infer_search_action(title):
    if not isinstance(title, str) or not title.strip():
        return pd.NA
    tokens = title.split()
    first_two = " ".join(tokens[:2])
    if first_two in known_search_actions:
        return first_two
    return tokens[0]


def query_from_title(title, search_action):
    if not isinstance(title, str) or search_action not in known_search_actions:
        return pd.NA
    remainder = title[len(search_action) :].strip()
    # English Takeout titles commonly say "Searched for ...".
    if search_action == "Searched":
        remainder = re.sub(r"^for\s+", "", remainder, flags=re.IGNORECASE).strip()
    return remainder if remainder else pd.NA


search_histories["search_action"] = search_histories["search_title"].apply(infer_search_action)
search_histories["is_known_search_action"] = search_histories["search_action"].isin(
    known_search_actions
)
search_histories["is_search_query"] = (
    search_histories["is_known_search_action"]
    & search_histories["has_standard_search_url"]
)
search_histories["query_from_title"] = search_histories.apply(
    lambda row: query_from_title(row["search_title"], row["search_action"]),
    axis=1,
)

search_histories[
    ["search_title", "search_action", "query_from_title", "query_from_url", "is_search_query"]
].head()

## Diagnostics

These checks document what happened during cleaning. For the mock dataset, we expect 120 combined search-history rows and 90 retained search-query rows.

In [ ]:
search_query_histories = (
    search_histories.loc[search_histories["is_search_query"]]
    .sort_values(["Participant ID", "time"], ascending=[True, False])
    .reset_index(drop=True)
)

diagnostics = {
    "donor_files_loaded": len(search_files),
    "combined_search_history_rows": len(search_histories),
    "search_query_rows": len(search_query_histories),
    "standard_search_url_rows": int(search_histories["has_standard_search_url"].sum()),
    "known_search_action_rows": int(search_histories["is_known_search_action"].sum()),
    "unrecognized_action_rows": int((~search_histories["is_known_search_action"]).sum()),
    "time_min": search_histories["time"].min(),
    "time_max": search_histories["time"].max(),
}

pd.Series(diagnostics)

In [ ]:
pd.DataFrame(
    {
        "all_search_rows": search_histories.groupby("Participant ID").size(),
        "search_query_rows": search_query_histories.groupby("Participant ID").size(),
    }
).fillna(0).astype(int)

In [ ]:
action_diagnostics = (
    search_histories.assign(
        search_action=search_histories["search_action"].fillna("UNRECOGNIZED")
    )
    .groupby(["search_action", "is_known_search_action"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

action_diagnostics

## Save Outputs

`search_histories.csv` keeps all standardized rows from the search-history exports. `search_query_histories.csv` keeps the rows that this notebook can confidently treat as genuine YouTube search queries.

In [ ]:
search_export_columns = [
    "Participant ID",
    "time",
    "header",
    "search_title",
    "search_action",
    "is_known_search_action",
    "is_search_query",
    "query_from_title",
    "query_from_url",
    "url",
    "details",
    "description",
]
search_query_columns = [
    "Participant ID",
    "time",
    "search_title",
    "search_action",
    "query_from_title",
    "query_from_url",
    "url",
]

search_histories_export = (
    search_histories.loc[:, search_export_columns]
    .sort_values(["Participant ID", "time"], ascending=[True, False])
    .reset_index(drop=True)
)
search_query_histories = search_query_histories.loc[:, search_query_columns]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
search_histories_export.to_csv(SEARCH_HISTORIES_PATH, index=False)
search_query_histories.to_csv(SEARCH_QUERY_HISTORIES_PATH, index=False)

saved_outputs = pd.DataFrame(
    {
        "table": ["search_histories", "search_query_histories"],
        "path": [SEARCH_HISTORIES_PATH, SEARCH_QUERY_HISTORIES_PATH],
        "rows": [len(search_histories_export), len(search_query_histories)],
    }
)
saved_outputs["path"] = saved_outputs["path"].apply(
    lambda output_path: output_path.relative_to(PROJECT_ROOT).as_posix()
)

saved_outputs

## Result

The workflow now has a combined donated search-history table and a narrower search-query table. The search-query table can be used later to estimate whether watch events occurred shortly after explicit YouTube searches.

### Stored Table Previews

The cells below show the first five rows of each dataframe saved as a CSV.

In [ ]:
stored_search_tables = {
    "search_histories.csv": search_histories_export,
    "search_query_histories.csv": search_query_histories,
}

for file_name, dataframe in stored_search_tables.items():
    print(f"{file_name} ({len(dataframe)} rows)")
    display(dataframe.head(5))